# Visualization 1 - Baby Names Through Time

This notebook creates **one simple static visualization** for the first mini-project question:

- Which names stay popular for a long time?
- Which names become popular only briefly?
- How does popularity change from 1900 to 2020?
- Do French baby names move in waves over time?

Instead of a bump chart, this notebook uses **small multiples** built from a small set of **signature names** drawn from different periods of the dataset.

In [13]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()
alt.renderers.enable('default')

RendererRegistry.enable('default')

In [14]:
# Load and clean the department-level dataset, then aggregate it to national level.
names = pd.read_csv('dpt2020.csv', sep=';')
names = names[(names['preusuel'] != '_PRENOMS_RARES') & (names['dpt'] != 'XX') & (names['annais'] != 'XXXX')].copy()
names['annais'] = names['annais'].astype(int)
names['nombre'] = names['nombre'].astype(int)

national = names.groupby(['annais', 'preusuel'], as_index=False)['nombre'].sum()

# Keep only the years 2000-2020, then compute the top 10 names of each year.
rank_data = national.copy()
rank_data = rank_data[(rank_data['annais'] >= 2000) & (rank_data['annais'] <= 2020)].copy()
rank_data['rank'] = rank_data.groupby('annais')['nombre'].rank(method='first', ascending=False)
rank_data = rank_data[rank_data['rank'] <= 10].copy()

# A helper field for cleaner labels.
rank_data['rank'] = rank_data['rank'].astype(int)
rank_data['rank_label'] = rank_data['rank'].astype(str)
rank_data = rank_data.sort_values(['preusuel', 'annais']).copy()

# Break lines when a name leaves the top 10 for one or more years.
rank_data['prev_year'] = rank_data.groupby('preusuel')['annais'].shift()
rank_data['new_segment'] = ((rank_data['prev_year'].isna()) | ((rank_data['annais'] - rank_data['prev_year']) > 1)).astype(int)
rank_data['segment'] = rank_data.groupby('preusuel')['new_segment'].cumsum()
rank_data['line_group'] = rank_data['preusuel'] + '_' + rank_data['segment'].astype(str)

rank_data.head()

,annais,preusuel,nombre,rank,rank_label,prev_year,new_segment,segment,line_group
218291,2014,ADAM,4549,8,8,NaN,1,1,ADAM_1
222885,2015,ADAM,4525,6,6,2014.0,0,1,ADAM_1
227325,2016,ADAM,4650,4,4,2015.0,0,1,ADAM_1
231786,2017,ADAM,4165,6,6,2016.0,0,1,ADAM_1
236198,2018,ADAM,3885,8,8,2017.0,0,1,ADAM_1


In [15]:
lines = alt.Chart(rank_data).mark_line(strokeWidth=1.6).encode(
    x=alt.X('annais:Q', title='Year', scale=alt.Scale(domain=[2000, 2020], nice=False), axis=alt.Axis(values=list(range(2000, 2021)), format='d', labelAngle=90)),
    y=alt.Y('rank_label:O', title='Rank within the year', sort=['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']),
    color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='tableau20')),
    detail='line_group:N',
    order=alt.Order('annais:Q'),
    tooltip=[
        alt.Tooltip('annais:Q', title='Year'),
        alt.Tooltip('preusuel:N', title='Name'),
        alt.Tooltip('rank:Q', title='Rank'),
        alt.Tooltip('nombre:Q', title='Births')
    ]
)

points = alt.Chart(rank_data).mark_circle(size=22).encode(
    x='annais:Q',
    y=alt.Y('rank_label:O', sort=['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']),
    color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='tableau20')),
    tooltip=[
        alt.Tooltip('annais:Q', title='Year'),
        alt.Tooltip('preusuel:N', title='Name'),
        alt.Tooltip('rank:Q', title='Rank'),
        alt.Tooltip('nombre:Q', title='Births')
    ]
)

labels = alt.Chart(rank_data).mark_text(fontSize=7, dy=-6).encode(
    x='annais:Q',
    y=alt.Y('rank_label:O', sort=['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']),
    color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='tableau20')),
    text='preusuel:N'
)

(lines + points + labels).properties(
    width=1000,
    height=360,
    title='Top 10 Baby Names of Each Year in France (2000-2020)'
)

alt.LayerChart(...)

## Why this visualization fits the assignment

- It shows the **top 10 names of each year** directly.
- The connecting lines make it possible to follow how names move up and down over time.
- The text labels on each point make the ranking readable without relying only on color.
- It is simple enough for a first implementation and can be refined later.
- It is different from:
  - a bump chart,
  - a top-10 ranking chart in 2020,
  - a 2000-2020 line chart of the top 5 names in 2020.